# 03 한국어 금융 말뭉치 MLP

## 목적
이전 MLP를 더 긴 한국어 금융/인공지능 말뭉치에 적용합니다. 슬라이딩 윈도우와 도메인/데이터 변화, fixed-context limitation을 확인합니다.

In [1]:
from pathlib import Path
import sys, torch

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.tokenizer import CharTokenizer, load_text
from src.baselines import MLPNextChar

torch.manual_seed(42)
text = load_text(str(ROOT / 'data/korean_finance_corpus.txt'))
tokenizer = CharTokenizer(text)
ids = tokenizer.encode(text)
print('corpus chars:', len(text), 'vocab_size:', tokenizer.vocab_size)
print(text[:80])

corpus chars: 2711 vocab_size: 370
인공지능은 금융 시장을 이해하는 새로운 도구가 될 수 있다.
하지만 좋은 도구가 되려면 데이터의 한계와 사회적 책임을 함께 배워야 한다.
이 작


## 핵심 코드
도메인이 한국어 금융/기술 문장으로 바뀌면 어휘와 자주 등장하는 패턴도 바뀝니다. 슬라이딩 윈도우는 긴 말뭉치에서 많은 `(고정 문맥, 다음 문자)` 예제를 만듭니다.

In [2]:
block_size = 3
limit = 180
x = torch.tensor([ids[i:i+block_size] for i in range(limit)], dtype=torch.long)
y = torch.tensor([ids[i+block_size] for i in range(limit)], dtype=torch.long)
print('first window:', tokenizer.decode(x[0].tolist()), '->', tokenizer.decode([int(y[0])]))
print('x shape:', tuple(x.shape), 'y shape:', tuple(y.shape))

first window: 인공지 -> 능
x shape: (180, 3) y shape: (180,)


In [3]:
model = MLPNextChar(tokenizer.vocab_size, block_size=3, n_embd=16, hidden=64)
opt = torch.optim.AdamW(model.parameters(), lr=0.03)
for _ in range(12):
    logits, loss, emb, flat = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
print('embedding shape:', tuple(emb.shape))
print('logits shape:', tuple(logits.shape))
print('short demo loss:', round(float(loss), 4))

embedding shape: (180, 3, 16)
logits shape: (180, 370)
short demo loss: 0.1991


/tmp/ipykernel_68614/327017601.py:10: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:838.)
  print('short demo loss:', round(float(loss), 4))


## 이 단계의 한계
고정 문맥 MLP는 마지막 세 글자 밖의 정보를 버립니다. 문장 앞에서 나온 금융 주제나 책임 기술 맥락을 뒤쪽 예측에 직접 사용할 수 없습니다.

## 다음 단계가 해결하는 점
다음 노트북은 매 위치마다 다음 문자를 예측하는 sequence language modeling 형태로 바꾸고 token embedding과 positional embedding을 사용합니다.